In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import hvplot.xarray
import holoviews as hv
import cmocean
from IPython.display import display

hv.extension("bokeh")

In [2]:
PARAM_CMAPS: dict[str, str] = {
    "sea_water_temperature":             "cmo.thermal",
    "sea_water_practical_salinity":      "cmo.haline",
    "corrected_dissolved_oxygen":        "cmo.oxy",
    "sea_water_density":                 "cmo.dense",
    "salinity_corrected_nitrate":        "cmo.matter",
    "nitrate_concentration":             "cmo.matter",
    "ph_seawater":                       "cmo.matter",
    "pco2_seawater":                     "cmo.matter",
    "partial_pressure_co2_ssw":          "cmo.matter",
    "fluorometric_chlorophyll_a":        "cmo.algae",
    "optical_backscatter":               "cmo.turbid",
    "flcdr_x_mmp_cds_fluorometric_cdom": "cmo.turbid",
}

DEFAULT_CMAP = "viridis"


def quadmesh(ds: xr.Dataset, param: str, width: int = 900, height: int = 350, **kwargs) -> hv.element.Element:
    da = ds[param].where(np.isfinite(ds[param]))
    return da.hvplot.quadmesh(
        x="time",
        y="sea_water_pressure",
        colorbar=True,
        flip_yaxis=True,
        cmap=PARAM_CMAPS.get(param, DEFAULT_CMAP),
        width=width,
        height=height,
        title=param.replace("_", " "),
        **kwargs,
    )


def check_infs(ds: xr.Dataset, params: list[str] | None = None) -> None:
    """Scan a dataset for inf values and report where they occur. Runs a full compute per param."""
    params = params if params is not None else list(ds.data_vars)
    for param in params:
        da = ds[param]
        inf_mask = np.isinf(da).compute()
        n_inf = int(inf_mask.sum())
        if n_inf:
            times = da.time.values[inf_mask.any(axis=list(inf_mask.dims).index("sea_water_pressure"))]
            t0 = pd.Timestamp(times[0]).isoformat()
            t1 = pd.Timestamp(times[-1]).isoformat()
            print(f"  {param}: {n_inf:,} inf values across {len(times)} time steps ({t0} → {t1})")
        else:
            print(f"  {param}: no infs")


def explore(
    file: str,
    params: list[str] | None = None,
    time_slice: slice | None = None,
) -> None:
    """Load a binned zarr and display a quadmesh for each parameter."""
    ds = xr.open_zarr(file)
    if time_slice:
        ds = ds.sel(time=time_slice)
    params_to_plot = params if params is not None else list(ds.data_vars)
    print(params_to_plot)
    print(f"{file}")
    print(f"time: {str(ds.time.values[0])[:10]} → {str(ds.time.values[-1])[:10]}  |  "
          f"pressure: {float(ds.sea_water_pressure.min()):.0f}–{float(ds.sea_water_pressure.max()):.0f} dbar  |  "
          f"vars: {list(ds.data_vars)}")
    for param in params_to_plot:
        display(quadmesh(ds, param))

In [3]:
# Run this cell to scan for inf values in a file (slow — computes the full array per param)
# check_infs(xr.open_zarr("../data/binned/oregon_shelf_profiles_20150803_20260529_qf4_binned_24h.zarr"))

In [12]:
# explore(
#     "../data/binned/axial_base_profiles_20150107_20260528_qf4_binned_24h.zarr",
#     # params=["sea_water_temperature", "ph_seawater"],
#     # time_slice=slice("2020", "2022"),
# )

In [9]:
# explore(
#     "../data/binned/oregon_shelf_profiles_20150803_20260529_qf4_binned_24h.zarr",
#     # params=["sea_water_temperature", "ph_seawater"],
#     # time_slice=slice("2020", "2022"),
# )

In [10]:
# explore(
#     "../data/binned/slope_base_profiles_20150709_20260528_qf4_binned_24h.zarr",
#     # params=["sea_water_temperature", "ph_seawater"],
#     # time_slice=slice("2020", "2022"),
# )

In [7]:
ds = xr.open_zarr("../data/binned/oregon_shelf_profiles_20150803_20260529_qf4_binned_24h.zarr")

In [8]:
ds

<xarray.Dataset> Size: 44MB
Dimensions:                       (time: 3953, sea_water_pressure: 200)
Coordinates:
  * time                          (time) datetime64[ns] 32kB 2015-08-03 ... 2...
  * sea_water_pressure            (sea_water_pressure) float64 2kB 0.0 ... 199.0
Data variables:
    corrected_dissolved_oxygen    (time, sea_water_pressure) float64 6MB ...
    pco2_seawater                 (time, sea_water_pressure) float64 6MB ...
    ph_seawater                   (time, sea_water_pressure) float64 6MB ...
    salinity_corrected_nitrate    (time, sea_water_pressure) float64 6MB ...
    sea_water_density             (time, sea_water_pressure) float64 6MB ...
    sea_water_practical_salinity  (time, sea_water_pressure) float64 6MB ...
    sea_water_temperature         (time, sea_water_pressure) float64 6MB ...
Attributes: (12/62)
    AssetManagementRecordLastModified:  2026-05-19T15:54:43.332000
    AssetUniqueID:                      ATOSU-66662-00007
    Conventions:                        CF-1.6
    Description:                        CTD Profiler: CTDPF Series A
    FirmwareVersion:                    Not specified.
    Manufacturer:                       Sea-Bird Electronics
    ...                                 ...
    stream:                             ctdpf_sbe43_sample
    subsite:                            CE04OSPS
    summary:                            Dataset Generated by Stream Engine fr...
    time_coverage_end:                  2026-05-29T10:08:46.330042368
    time_coverage_start:                2014-11-05T21:30:49.640057856
    title:                              Data produced by Stream Engine versio...